# Bitcoin Exploratory Data Analysis

This notebook documents the reproducible exploratory data analysis for Domain 1 - Financial Time Series. It prepares the minute-level BTC/USD data for the downstream forecasting notebooks without training any models.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data
from src.plots import plot_time_series

## 2. Load Raw Dataset

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"

raw_df = load_bitcoin_data(DATA_PATH)
raw_df.head()

In [ ]:
raw_summary = pd.DataFrame(
    {
        "Metric": [
            "Rows",
            "Columns",
            "Start timestamp",
            "End timestamp",
            "Duplicate timestamps",
            "Missing close values",
        ],
        "Value": [
            len(raw_df),
            raw_df.shape[1],
            raw_df["Timestamp"].min(),
            raw_df["Timestamp"].max(),
            raw_df["Timestamp"].duplicated().sum(),
            raw_df["Close"].isna().sum(),
        ],
    }
)
raw_summary

## 3. Prepare Daily Bitcoin Series

In [ ]:
df_daily = prepare_daily_bitcoin_data(raw_df)
df_daily.head()

In [ ]:
daily_summary = pd.DataFrame(
    {
        "Metric": [
            "Daily rows",
            "Start date",
            "End date",
            "Missing daily close values",
            "Minimum close",
            "Maximum close",
        ],
        "Value": [
            len(df_daily),
            df_daily.index.min(),
            df_daily.index.max(),
            df_daily["Close"].isna().sum(),
            df_daily["Close"].min(),
            df_daily["Close"].max(),
        ],
    }
)
daily_summary

## 4. Visualise Daily Close

In [ ]:
plot_time_series(df_daily, column="Close");

## 5. Returns and Volatility

In [ ]:
daily_returns = df_daily["Close"].pct_change()
rolling_volatility = daily_returns.rolling(window=30).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
daily_returns.plot(ax=axes[0], linewidth=0.8, title="Bitcoin Daily Returns")
rolling_volatility.plot(ax=axes[1], linewidth=1.2, title="30-Day Rolling Return Volatility")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()

## 6. Chronological Train-Test Split

In [ ]:
target = df_daily["Close"]
split_idx = int(len(target) * 0.8)
train = target.iloc[:split_idx]
test = target.iloc[split_idx:]

split_summary = pd.DataFrame(
    {
        "Split": ["Train", "Test"],
        "Start": [train.index.min(), test.index.min()],
        "End": [train.index.max(), test.index.max()],
        "Length": [len(train), len(test)],
    }
)
split_summary

In [ ]:
assert train.index.max() < test.index.min()
assert len(train) + len(test) == len(target)
assert test.index.is_monotonic_increasing
assert train.index.is_monotonic_increasing

## 7. EDA Findings

- The physically present Bitcoin dataset is minute-level OHLCV data.
- The forecasting task uses the daily resampled `Close` series.
- The downstream notebooks use an 80/20 chronological split with no train-test overlap.
- Bitcoin prices are strongly non-stationary, so raw-price neural models require careful diagnostics for range compression and over-smoothing.
- Volatility clustering motivates separate robustness checks for low-volatility, high-volatility, major upward-move, and major downward-move regimes.